## 1. Dataset Background

The data was collected from DAEWOO Steel Co. Ltd. in South Korea and published by Sathishkumar, V. E. and others. It comes from a smart small-scale steel industry. The dataset covers a one-year period from the 1st of January to the 31st of December 2018. Samples were taken every 15 minutes which gives 35040 rows. Each row represents a set of electrical and operating measurements recorded during a specific 15-minute interval.

## 2. Dataset Structure

In [44]:
# importing necessary libraries
import pandas as pd

In [ ]:
# loading dataset
df = pd.read_csv('../data/Steel_industry_data.csv')

In [46]:
# checking the number of rows and columns in the dataset
df.shape

(35040, 11)

In [47]:
# checking the names of the columns in the dataset
df.columns.to_list()

['date',
 'Usage_kWh',
 'Lagging_Current_Reactive.Power_kVarh',
 'Leading_Current_Reactive_Power_kVarh',
 'CO2(tCO2)',
 'Lagging_Current_Power_Factor',
 'Leading_Current_Power_Factor',
 'NSM',
 'WeekStatus',
 'Day_of_week',
 'Load_Type']

In [48]:
# checking the data type of each column
df.dtypes

date                                        str
Usage_kWh                               float64
Lagging_Current_Reactive.Power_kVarh    float64
Leading_Current_Reactive_Power_kVarh    float64
CO2(tCO2)                               float64
Lagging_Current_Power_Factor            float64
Leading_Current_Power_Factor            float64
NSM                                       int64
WeekStatus                                  str
Day_of_week                                 str
Load_Type                                   str
dtype: object

In [49]:
#checking for missing values
df.isna().sum()

date                                    0
Usage_kWh                               0
Lagging_Current_Reactive.Power_kVarh    0
Leading_Current_Reactive_Power_kVarh    0
CO2(tCO2)                               0
Lagging_Current_Power_Factor            0
Leading_Current_Power_Factor            0
NSM                                     0
WeekStatus                              0
Day_of_week                             0
Load_Type                               0
dtype: int64

In [50]:
# checking for duplicate rows
df.duplicated().sum() 

np.int64(0)

Findings: The dataset was confirmed to contain 35040 rows and 11 columns with the features having mixed data types of strings, integers and floats. The variables also fall under different types of information: Temporal (date, NSM), continuous numerical (energy, reactive power, power factor, CO2) and categorical (Week status, day of week, load type). There were no missing values or duplicated rows found.
One point requiring attention is that the date feature is stored as a string rather than a datetime object.

## 3. Understanding the Variables

| Variable | What it represents | Unit | Data Type | Initial Role | Potential Leakage|
| :- | :-: | :-: | :-: | :-: | :-: |
| date | the date and time the observation was taken | N/A | temporal | provides chronological order, allows for investigation of temporal patterns, and for time-based forecasting | No |
| Usage_kWh | energy consumed in a particular 15 min interval | kWh | numerical | possible numerical target variable | Target |
| Lagging_Current_Reactive.Power_kVarh | reactive energy associated with loads where current lags the voltage, typically associated with inductive equipment | kVarh| numerical | possible predictor of usage kwh | Investigate |
| Leading_Current_Reactive_Power_kVarh | reactive energy associated with loads where current leads the voltage, typically associated with capacitive behaviour | kVarh | numerical | possible predictor of usage kwh | Investigate |
| CO2(tCO2) | CO₂ emissions from energy used | tonnes | numerical | potentially derived from energy consumption - investigate before using as a predictor | Potentially, yes |
| Lagging_Current_Power_Factor | indicates how effectively apparent power is being converted into useful real power | dimensionless | numerical | possible predictor of usage kwh | Investigate |
| Leading_Current_Power_Factor | indicates how effectively apparent power is being converted into useful real power | dimensionless | numerical | possible predictor of usage kwh | Investigate |
| NSM | how many seconds past midnight the observation was taken | seconds | temporal | helps to indicate how energy is consumed over time in a one-day period | No |
| WeekStatus | whether it is weekend or weekday | N/A | categorical | used to show the difference between energy usage on weekends and weekdays | No |
| Day_of_week | identifies the day of the week | N/A | categorical | allows to investigate differences in energy consumption across days | No |
| Load_Type  | categorical classification of the operating/load condition | N/A | categorical | used to show how different levels of load correspond with energy usage | Investigate |

Usage_kWh was identified as the target because the primary goal of the project is to analyse energy consumption and forecast future energy usage. Since Usage_kWh represents the energy consumed during each 15-minute interval, it directly represents the quantity being analysed and predicted.

## 4. The Time Dimension

In [51]:
# converting the date column to a datetime object
df['date'] = pd.to_datetime(df['date'], format = '%d/%m/%Y %H:%M')
df.dtypes

date                                    datetime64[us]
Usage_kWh                                      float64
Lagging_Current_Reactive.Power_kVarh           float64
Leading_Current_Reactive_Power_kVarh           float64
CO2(tCO2)                                      float64
Lagging_Current_Power_Factor                   float64
Leading_Current_Power_Factor                   float64
NSM                                              int64
WeekStatus                                         str
Day_of_week                                        str
Load_Type                                          str
dtype: object

In [52]:
# checking for the earliest and latest timestamp
earliest = df['date'].min()
print(earliest)
latest = df['date'].max()
print(latest)

2018-01-01 00:00:00
2018-12-31 23:45:00


In [124]:
# checking the interval between date samples
df['date'].diff().unique()

<TimedeltaArray>
[NaT, '0 days 00:15:00', '-1 days +00:15:00', '1 days 00:15:00']
Length: 4, dtype: timedelta64[us]

In [125]:
# checking the unusual time intervals
df['time_diff'] = df['date'].diff()
unusual = df[df['time_diff'] != pd.Timedelta(minutes=15)]
unusual[['date', 'time_diff']]

,date,time_diff
0,2018-01-01 00:15:00,NaT
95,2018-01-01 00:00:00,-1 days +00:15:00
96,2018-01-02 00:15:00,1 days 00:15:00
191,2018-01-02 00:00:00,-1 days +00:15:00
192,2018-01-03 00:15:00,1 days 00:15:00
...,...,...
34847,2018-12-29 00:00:00,-1 days +00:15:00
34848,2018-12-30 00:15:00,1 days 00:15:00
34943,2018-12-30 00:00:00,-1 days +00:15:00
34944,2018-12-31 00:15:00,1 days 00:15:00


In [131]:
# checking if the number of observations for each day is correct (24x4 =96)
df.head(96)

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,time_diff
0,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load,NaT
1,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load,0 days 00:15:00
2,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load,0 days 00:15:00
3,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load,0 days 00:15:00
4,2018-01-01 01:15:00,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load,0 days 00:15:00
...,...,...,...,...,...,...,...,...,...,...,...,...
91,2018-01-01 23:00:00,3.53,3.74,0.0,0.0,68.64,100.0,82800,Weekday,Monday,Light_Load,0 days 00:15:00
92,2018-01-01 23:15:00,3.53,3.42,0.0,0.0,71.82,100.0,83700,Weekday,Monday,Light_Load,0 days 00:15:00
93,2018-01-01 23:30:00,3.24,2.95,0.0,0.0,73.94,100.0,84600,Weekday,Monday,Light_Load,0 days 00:15:00
94,2018-01-01 23:45:00,3.67,3.96,0.0,0.0,67.97,100.0,85500,Weekday,Monday,Light_Load,0 days 00:15:00


In [92]:
df['date'].duplicated().sum()

np.int64(0)

The dataset contains 96 observations for each day. However, it was observed that the observations are not strictly chronological. The 00:00 observation is placed after the 23:45 observation causing an inconsistent transition. The NSM confirms this pattern. Chronological reordering is needed before time-series analysis and forecasting. There is also no duplicated date/time in the dataset.
